<a href="https://colab.research.google.com/github/Orliluq/ONE-AI-FOR-TECH-RAG/blob/main/Inteligencia_de_Datos_y_RAG_Avanzado_ONE_AI_FOR_TECH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-google-genai google-generativeai

In [ ]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
import os

# Suponiendo que tienes una Clave de API almacenada en una variable de entorno
api_key = os.getenv('GEMINI_API_KEY')

# Usa la Clave de API para acceder a un servicio
print(f"Usando la Clave de API: {api_key}")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=GEMINI_API_KEY
)

In [ ]:
respuesta = llm.invoke("¿Qué es el RAG en Inteligencia Artificial?")

In [ ]:
respuesta.content

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# Asumiendo que GEMINI_API_KEY ya ha sido definida en una celda anterior.
# Creando una instancia del LLM de Google
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", # Cambiado de "gemini-pro" a "gemini-2.5-flash"
    temperature=0,
    google_api_key=GEMINI_API_KEY
)

# Usando el LLM para generar texto
prompt_text = "Explica qué es inteligencia artificial."
response = llm.invoke([HumanMessage(content=prompt_text)]) # Wrap HumanMessage in a list

# Exibiendo la respuesta generada por el LLM
print(response.content)

In [ ]:
PROMPT_TRIAJE = """
Eres un especialista en triaje del Service Desk para politicas internas.
Dado el mensaje del usuario, devuelve SÓLO un JSON con:\n
{\n
    "decision": "AUTO_RESOLVER" | "PEDIR_INFO" | "ABRIR_TICKET",\n
    "urgency": "BAJA" | "MEDIANA" | "ALTA",\n
    "missing_fields": ["..."]\n
}\n
Reglas:\n
- **AUTO_RESOLVER**: Preguntas claras sobre las reglas o procedimientos descritos en las politicas (Ej.: "¿Puedo reembolsar el internet para mi oficina en casa?").\n
- **PEDIR_INFO**: Mensajes imprecisos o sin información para identificar el tema o el contexto (Ej.: "Necesito ayuda con una politica").\n
- **ABRIR_TICKET**: Solicitudes de excepciones, autorización, aprobación o acceso especial, o cuando el usuario solicita explicitamente abrir un ticket (Ej.: "Quiero una excepción para trabajar remotamente durante 5 dias").\n
Analiza el mensaje y decide la acción más adecuada.
"""

In [ ]:
from typing import Literal, List, Dict

In [ ]:
from pydantic import BaseModel, Field

In [ ]:
class TriajeOut(BaseModel):
    decision: Literal["AUTO_RESOLVER", "PEDIR_INFO", "ABRIR_TICKET"]
    urgencia: Literal["BAJA", "MEDIANA", "ALTA"]
    campos_faltantes: List[str] = Field(default_factory=list)

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:
chain_de_triaje = llm.with_structured_output(TriajeOut)

def triaje(mensaje: str) -> Dict:
    salida: TriajeOut = chain_de_triaje.invoke(
        [
            SystemMessage(content=PROMPT_TRIAJE),
            HumanMessage(content=mensaje)
        ]
    )
    return salida.model_dump()

In [ ]:
mensajes_de_prueba = [
    "¿Puedo obtener un reembolso por el internet de mi home office?",
    "Quiero una excepción para teletrabajar durante 5 días.",
    "¿Cómo funciona la política de comidas para viajes?",
    "¿Existe una política para anticipos de vacaciones?",
    "¿Quién fue Napoleón Bonaparte?"
]

In [ ]:
for pregunta in mensajes_de_prueba:
    r = triaje(pregunta)
    print(f"{pregunta} -> {r}")

## Describiendo los documentos de recursos humanos

In [ ]:
!pip install -q langchain_community faiss-cpu langchain-text-splitters pymupdf

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

docs = []

for n in Path("/content/").glob("*.pdf"):
    try:
        loader = PyMuPDFLoader(str(n))
        docs.extend(loader.load())
        print(f"Archivo cargado: {n.name}")
    except Exception as e:
        print(f"Error cargando archivo: {n.name}: {e}")

print(f"Total de documentos cargados: {len(docs)}")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)

In [ ]:
docs_splits = splitter.split_documents(docs)

In [ ]:
for chunk in docs_splits:
    print(chunk)
    print("-----------------")

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [ ]:
modelo_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GEMINI_API_KEY
)

In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
vectorstore = FAISS.from_documents(docs_splits, modelo_embeddings)

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.3, "k": 4}
)

In [ ]:
# Suponemos que tenemos una función para buscar documentos
def buscar_documentos(pregunta):
    # Esta función devuelve documentos relevantes
    documentos = {
        "pandas": "Los pandas se alimentan principalmente de bambú."
    }
    return documentos.get(pregunta.lower(), "")

# Función para generar respuesta usando GPT
def generar_respuesta(pregunta):
    # Primero, recuperamos documentos relevantes
    documento = buscar_documentos(pregunta)

    # Luego, usamos el modelo de IA para generar una respuesta
    if documento:
        respuesta = f"Basado en información interna, sabemos que {documento}"
    else:
        respuesta = "Lo siento, no tengo suficiente información para responder a eso."

    return respuesta

# Ejemplo de uso
pregunta = "¿Qué comen los pandas?"
print(generar_respuesta(pregunta))

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
# from langchain.chains.combine_documents import create_stuff_documents_chain # This import is no longer needed and causes a ModuleNotFoundError

In [ ]:
prompt_rag = ChatPromptTemplate(
    [
        ("system",
            """Eres el especialista en RR.HH. de la empresa Carraro Desarrollo de Software.
            Responde siempre utilizando los conocimientos de las bases de datos pasadas a ti.
            Si no hay informacion sobre la pregunta en los datos, responde solo 'No lo se'.
            """
        ),
        ("human", "Contexto: {context}\nPregunta del empleado: {question}") # Changed {input} to {question}
    ]
)

# document_chain = create_stuff_documents_chain(llm, prompt_rag) # This is no longer needed and caused an error

In [ ]:
def busqueda_de_respuestas_RAG(pregunta) -> Dict:
  # The 'retriever' variable in the global scope will now refer to the one
  # associated with the Chroma vector store (defined in cell LsZXp5N89aAB).
  documentos_relacionados = retriever.invoke(pregunta)

  if not documentos_relacionados:
    return {
        "respuesta": "No lo sé",
        "citaciones": [],
        "documentos_encontrados": False
    }

  # Use the 'rag_chain' (defined in cell 3CKgOUx99tVA) which is functional.
  # It handles the context formatting and LLM invocation internally.
  answer = rag_chain.invoke(pregunta)

  if answer.rstrip(".!?") == "No lo sé":
      return {
          "respuesta": "No lo sé",
          "citaciones": [],
          "documentos_encontrados": False
      }

  return {
      "respuesta": answer,
      "citaciones": documentos_relacionados, # Use the separately retrieved documents for citations
      "documentos_encontrados": True
  }

In [ ]:
busqueda_de_respuestas_RAG("¿Puedo obtener un reembolso por el internet de mi home office?")

In [ ]:
mensajes_de_prueba = [
    "¿Puedo obtener un reembolso por el internet de mi home office?",
    "Quiero una excepción para teletrabajar durante 5 días.",
    "¿Cómo funciona la política de comidas para viajes?",
    "¿Existe una politica para anticipos de vacaciones?",
    "¿Quién fue Napoleon Bonaparte?"
]

In [ ]:
r = busqueda_de_respuestas_RAG("¿Puedo obtener un reembolso por el internet de mi home office?")
print(r)

In [ ]:
print(f"Número de citaciones para la última respuesta: {len(r['citaciones'])}")

In [ ]:
for pregunta in mensajes_de_prueba:
    respuesta_RAG = busqueda_de_respuestas_RAG(pregunta)

    print(f"\nPREGUNTA: {pregunta}")
    print(f"RESPUESTA: {respuesta_RAG['respuesta']}")
    print(f"DOCUMENTOS ENCONTRADOS: {respuesta_RAG['documentos_encontrados']}")

    if respuesta_RAG['documentos_encontrados']:
        for i, citacion in enumerate(respuesta_RAG['citaciones'], start=1):
            print(f"\nCITACION {i}:")
            print(
                f"Camino del documento: "
                f"{citacion.metadata.get('file_path', 'No disponible')}"
            )
            print(
                f"Contenido: "
                f"{citacion.page_content.replace(chr(10), ' ')}"
            )

    print("\n" + "=" * 80)

In [ ]:
import time

# Nos aseguramos de que el rag_chain use el LLM local para evitar errores de cuota
# hf_llm fue definido en la celda fe500b6a
if 'hf_llm' in globals():
    llm = hf_llm

for pregunta in mensajes_de_prueba:
    try:
        print(f"PROCESANDO: {pregunta}")
        respuesta_RAG = busqueda_de_respuestas_RAG(pregunta)

        print(f"PREGUNTA: {pregunta}")
        print(f"RESPUESTA: {respuesta_RAG['respuesta']}")
        print(f"DOCUMENTOS ENCONTRADOS: {respuesta_RAG['documentos_encontrados']}")

        if respuesta_RAG['documentos_encontrados']:
            for i, citacion in enumerate(respuesta_RAG['citaciones']):
                print(f"  CITACION {i + 1}:")
                # Usamos .get() para evitar errores si la clave no existe
                source = citacion.metadata.get('file_path', citacion.metadata.get('source', 'Desconocido'))
                print(f"    Origen: {source}")
                print(f"    Contenido: {citacion.page_content[:200].replace('\n', ' ')}...")

        print("-" * 30)
        # Pequeña pausa para no saturar la salida si el modelo es muy rápido
        time.sleep(0.5)

    except Exception as e:
        print(f"Error al procesar la pregunta '{pregunta}': {e}")

# OTRO PLANTEAMIENTO

## Instalar dependencias

In [ ]:
# !pip install -q \
# langchain \
# langchain-openai \
# langchain-community \
# langchain-core \
# chromadb \
# pypdf \
# tiktoken \
# langchain-text-splitters # Commented out to prevent dependency conflicts

 ## Configurar API Key

In [ ]:
from google.colab import userdata
import os

os.environ["OPEN_AI_API_KEY"] = userdata.get("OPEN_AI_API_KEY")

## Subir PDFs

In [ ]:
from google.colab import files

uploaded = files.upload()

## Cargar documentos

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

documents = []

for file_name in uploaded.keys():
    loader = PyPDFLoader(file_name)
    docs = loader.load()

    documents.extend(docs)

print(f"Documentos cargados: {len(documents)}")

## Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Chunks creados: {len(chunks)}")

## Crear embeddings

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

documents = []

for file_name in uploaded.keys():
    loader = PyPDFLoader(file_name)
    docs = loader.load()

    documents.extend(docs)

print(f"Documentos cargados: {len(documents)}")

In [ ]:
from langchain_openai import OpenAIEmbeddings
from google.colab import userdata
from google.colab.userdata import SecretNotFoundError

try:
    # Attempt to retrieve the API key directly from Colab secrets
    # The context indicates the user might have set 'OPEN_AI_API_KEY' or 'openai_key'.
    # This code specifically looks for 'OPEN_AI_API_KEY'.
    openai_api_key = userdata.get("OPEN_AI_API_KEY")
except SecretNotFoundError:
    raise ValueError(
        "OpenAI API key not found. Please ensure you have set a Colab secret "
        "named 'OPEN_AI_API_KEY' (or 'openai_key' if that's what you intended to use, "
        "and adjust the code accordingly). Alternatively, set the 'OPENAI_API_KEY' "
        "environment variable (without the underscore in 'OPENAI_API_KEY' for LangChain)."
    )

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=openai_api_key  # Pass the key directly to the constructor
)

### Usar Embeddings de HuggingFace (gratuitos y locales)

In [ ]:
# Instalar las librerías necesarias para HuggingFace Embeddings y la versión recomendada por LangChain
# !pip install -q --force-reinstall langchain-huggingface sentence-transformers transformers # Commented out to prevent dependency conflicts

### Instalar librerías adicionales para ejecutar modelos locales (HuggingFace LLM)

In [ ]:
# `accelerate` y `bitsandbytes` son útiles para ejecutar modelos más grandes de forma eficiente.
# `torch` debería estar instalado por otras dependencias, pero asegúrate de que sea una versión compatible.
# !pip install -q accelerate bitsandbytes # Commented out to prevent dependency conflicts

In [ ]:
# Importar HuggingFaceEmbeddings de la nueva ubicación recomendada
from langchain_huggingface import HuggingFaceEmbeddings

# Inicializar HuggingFaceEmbeddings con un modelo pre-entrenado. Puedes elegir otros modelos si lo deseas.
# 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2' es un buen modelo multilingüe.
# Para modelos en español, considera 'hiiamsid/sentence_similarity_spanish_es'
# O 'distilbert-base-nli-stsb-mean-tokens' para inglés (requiere descargar el modelo)

hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Asignar la instancia de HuggingFaceEmbeddings a la variable 'embeddings'
# para que el siguiente paso de Chroma.from_documents la use.
embeddings = hf_embeddings

print("Embeddings de HuggingFace inicializados correctamente.")

In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("Base vectorial creada")

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

In [ ]:
query = "¿Cuál es el objetivo principal del documento?"

results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ---\n")
    print(doc.page_content[:500])

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Eres un asistente especializado en responder preguntas
usando exclusivamente el contexto proporcionado.

Reglas:

1. Responde únicamente con información presente en el contexto.
2. No inventes datos.
3. Si la respuesta no está en el contexto, responde:

"No encontré información suficiente en los documentos para responder esa pregunta."

Contexto:

{context}

Pregunta:

{question}

Respuesta:
""")

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=openai_api_key # Pasa la clave directamente al constructor
)

### Configurar un LLM de Hugging Face

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
import torch

# Define el nombre del modelo de Hugging Face que quieres usar
# TinyLlama/TinyLlama-1.1B-Chat-v1.0 es un modelo pequeño adecuado para pruebas en Colab
# Si tienes GPU y más RAM, puedes probar modelos más grandes como 'google/gemma-2b-it'
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Cargar el tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Cargar el modelo. Usa torch_dtype=torch.bfloat16 para mayor eficiencia si tu GPU lo soporta.
# Si tienes problemas de memoria, puedes intentar cargar con quantization_config (e.g., bitsandbytes)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

# Crear un pipeline de texto con el modelo y el tokenizer
pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    temperature=0.1,
    do_sample=False
    # max_new_tokens=512, # Limita la longitud de la respuesta generada
    # temperature=0.1,    # Ajusta para controlar la creatividad de la respuesta
)

# Inicializar el LLM de HuggingFace para LangChain
hf_llm = HuggingFacePipeline(pipeline=pipeline)

# Sobrescribir la variable `llm` para que el `rag_chain` use el nuevo LLM de Hugging Face
llm = hf_llm

print(f"LLM de Hugging Face '{model_name}' inicializado correctamente y configurado para la cadena RAG.")


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
# The format_docs function definition was moved to cell 3CKgOUx99tVA to ensure it's defined before rag_chain.

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt_rag # Changed prompt to prompt_rag
    | llm  # This will now correctly reference the HuggingFace LLM
    | StrOutputParser()
)

In [ ]:
docs = retriever.invoke(
    "¿Cada cuánto deben cambiarse las contraseñas?"
)

print(format_docs(docs))

In [ ]:
rag_chain.invoke(
    "¿Cuál es la política de licencia por maternidad?"
)